In [1]:
# data/NEXTIER_Internship_Bugs.md
#cell1:
import os
import shutil
import re
import json
from dotenv import load_dotenv
import shutil 


# LangChain Imports
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.documents import Document

# For Advanced RAG Data Foundation (Version 1.2+)
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import LocalFileStore
from langchain_classic.storage import create_kv_docstore


load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("🚨 OPENAI_API_KEY not found! Check your .env file.")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Nexteir_Second_Brain_Lab"

print("✅ Environment loaded. Tracing disabled/enabled as per config.")

/Users/victorloucii/agentic_ai_engineer/Unified-Knowledge-Agent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Environment loaded. Tracing disabled/enabled as per config.


In [2]:
#cell2:

# ==========================================
# 1. THE "SILVER" DATA FOUNDATION (Extract & Clean)
# ==========================================
md_path = "../data/NEXTIER_Internship_Bugs.md"

try:
    with open(md_path, "r", encoding="utf-8") as f:
        raw_md_text = f.read()
    print(f"📄 Loaded raw file successfully.")
except FileNotFoundError:
    raise FileNotFoundError(f"🚨 Markdown file not found at {md_path}.")

# --- THE MAGIC FIX: Convert // syntax to Markdown syntax ---
# 1. Convert "//problem 23", "//Problem23", etc., into "# Problem 23"
# (?i) makes it case-insensitive. \s* handles optional spaces.
clean_md_text = re.sub(r'(?i)//\s*problem\s*(\d*)', r'# Problem \1', raw_md_text)

# 2. (Optional) Convert "//useful command" into an H2 if you use those
clean_md_text = re.sub(r'(?i)//\s*useful\s*command', r'## Useful Command', clean_md_text)

# 3. Strip weird invisible Unicode artifacts
clean_md_text = clean_md_text.replace('\xa0', ' ')

# 4. Nuke the Base64 Images
# ==========================================
# This regex hunts down ![any_alt_text](data:image/...) and replaces it with a clean placeholder.
clean_md_text = re.sub(r'!\[.*?\]\(data:image\/[^;]+;base64,[^)]+\)', '[SCREENSHOT REMOVED]', clean_md_text)

print("🧹 Cleaned data, standardized headers, and purged Base64 images.")

print("🧹 Cleaned data and standardized headers.")

# ==========================================
# 2. STRUCTURE-AWARE SPLITTING
# ==========================================
headers_to_split_on = [
    ("#", "Header 1"),
    # ("##", "Header 2"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
parent_docs = markdown_splitter.split_text(clean_md_text)

# Check the actual count after the H2 split fix
print(f"📊 TOTAL UNIQUE PROBLEMS FOUND: {len(parent_docs)}")

# 👇 ADD THIS DEBUG LOOP 👇
print("\n🕵️‍♂️ FINDING THE IMPOSTER HEADERS:")
imposter_count = 0
for doc in parent_docs:
    header_name = doc.metadata.get("Header 1", "")
    if "Problem" not in header_name:
        imposter_count += 1
        print(f"[{imposter_count}] Extra Header: {header_name}")

# --- FAIL-SAFE CHECK ---
if len(parent_docs) == 0:
    raise ValueError("🚨 Splitter returned 0 documents! The regex didn't catch your headers. Check your file syntax.")

print(f"🧩 Split into {len(parent_docs)} Parent Documents (Sections/Problems).")

print("\n🕵️‍♂️ AUDITING THE PROBLEM HEADERS:")
problem_count = 0
for doc in parent_docs:
    header_name = doc.metadata.get("Header 1", "")
    if "Problem" in header_name:
        problem_count += 1
        print(f"[{problem_count}] {header_name}")

📄 Loaded raw file successfully.
🧹 Cleaned data, standardized headers, and purged Base64 images.
🧹 Cleaned data and standardized headers.
📊 TOTAL UNIQUE PROBLEMS FOUND: 147

🕵️‍♂️ FINDING THE IMPOSTER HEADERS:
[1] Extra Header: 
[2] Extra Header: **🔎 What Was Actually Happening (Root Cause Analysis)**
[3] Extra Header: **🧠 Why This Felt So Confusing**
[4] Extra Header: **✅ Final Stable Implementation (Correct Setup)**
[5] Extra Header: **🧠 The Big Lesson (Very Important)**
[6] Extra Header: **Bug Resolution Summary: React Navigation Reset Crash on Role-Based Stacks**
[7] Extra Header: **1️⃣ Parent screen controls horizontal spacing**
[8] Extra Header: **2️⃣ Recommended communities section**
[9] Extra Header: **3️⃣ The FlatList problem**
[10] Extra Header: **4️⃣ Why we add marginLeft**
[11] Extra Header: **5️⃣ Why use theme.spacing.md**
[12] Extra Header: **6️⃣ Why only for the first card**
[13] Extra Header: **Final layout (correct)**
[14] Extra Header: **Best Practice: Move React Nativ

In [3]:
#cell3:
# ==========================================
# 3. ADVANCED RETRIEVAL (Parent-Document Retrieval)
# ==========================================
persist_directory = "./chroma_db"
store_directory = "./docstore" 

# --- THE CRITICAL FIX: WIPE THE OLD GHOST DATA ---
# This ensures we start with a clean slate before rebuilding
if os.path.exists(persist_directory):
    shutil.rmtree(persist_directory)
    print("🗑️ Wiped old ChromaDB.")
if os.path.exists(store_directory):
    shutil.rmtree(store_directory)
    print("🗑️ Wiped old DocStore.")

# Swapped to Local Embeddings (Cost: $0)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Initialize the Vector Store
vectorstore = Chroma(
    collection_name="nexteir_internship",
    embedding_function=embeddings,
    persist_directory=persist_directory
)

# Initialize the raw file store (Saves bytes)
raw_store = LocalFileStore(store_directory)

# Automatically wrap the raw store so it can handle LangChain Documents
store = create_kv_docstore(raw_store)

# Initialize the Child Splitter
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

# Create the Parent Document Retriever
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    search_kwargs={"k": 15}, 
)

# --- REVISED INGESTION LOGIC ---
force_rebuild = True 

if force_rebuild:
    print("🏗️ Ingesting documents into Local Vector Store in batches...")
    
    batch_size = 20 
    
    for i in range(0, len(parent_docs), batch_size):
        batch = parent_docs[i : i + batch_size]
        retriever.add_documents(batch)
        print(f"✅ Processed batch {i // batch_size + 1} (Documents {i} to {i + len(batch)})...")
        
    print("🎉 All batches ingested successfully!")
else:
    print("📦 Using existing persistent database. (Cost: $0)")

# --- QUICK DIAGNOSTIC ---
print("\n🔍 [DEBUG] Testing Parent Retriever...")
res = retriever.invoke("problem 13")

if len(res) > 0:
    print(f"✅ Verified: Found {len(res)} full parent document(s).")
    print(f"📄 Preview metadata: {res[0].metadata}")
    print(f"📄 Preview content: {res[0].page_content[:150]}...")
else:
    print("❌ Failure: Search term not found.")

🗑️ Wiped old ChromaDB.
🗑️ Wiped old DocStore.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10438.63it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🏗️ Ingesting documents into Local Vector Store in batches...
✅ Processed batch 1 (Documents 0 to 20)...
✅ Processed batch 2 (Documents 20 to 40)...
✅ Processed batch 3 (Documents 40 to 60)...
✅ Processed batch 4 (Documents 60 to 80)...
✅ Processed batch 5 (Documents 80 to 100)...
✅ Processed batch 6 (Documents 100 to 120)...
✅ Processed batch 7 (Documents 120 to 140)...
✅ Processed batch 8 (Documents 140 to 147)...
🎉 All batches ingested successfully!

🔍 [DEBUG] Testing Parent Retriever...
✅ Verified: Found 13 full parent document(s).
📄 Preview metadata: {'Header 1': 'Problem 19'}
📄 Preview content: Here is a complete summary of the problems and the solutions we
implemented. You can save this as documentation for future reference!  
\-\--  
\# Bug...


In [4]:
#cell4:

@tool
def search_internship_history(query: str) -> str:
    """
    Search the Nexteir Internship logs. This is a comprehensive 'Second Brain' 
    containing debugging fixes (//problemXX), architectural decisions, 
    and useful CLI/Terminal commands (//useful command).
    """
    # 1. Multi-Query Expansion: Let the LLM "think" of better search terms
    # This is what makes it "Lazy Search" friendly.
    expansion_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    search_variants = expansion_llm.invoke(
        f"Generate 3 diverse search queries to find information about '{query}' "
        "in technical internship logs. Focus on keywords, problem IDs, and file paths. "
        "Respond with only the queries, one per line."
    ).content.split("\n")

    # Add the original query to the list
    all_queries = [query] + [q.strip() for q in search_variants if q.strip()]
    
    # 2. Execute a broad search across all variants
    all_docs = []
    for q in all_queries:
        all_docs.extend(retriever.invoke(q))

    # 3. Deduplicate (so we don't show the same chunk twice)
    # Convert dict_values to a list so it can be sorted and sliced
    unique_docs = list({doc.page_content: doc for doc in all_docs}.values())

    # 4. CUSTOM RE-RANKER (Singular/Plural 'issues' vs 'issue' fix)
    def score_doc(doc):
        text = doc.page_content.lower()
        score = 0
        # Split the query into words, ignore very short words
        keywords = [word.lower() for word in query.split() if len(word) > 2]
        
        for kw in keywords:
            # Basic singular/plural handling
            base = kw[:-1] if kw.endswith('s') else kw
            plural = base + 's'
            
            # Check for either form in the text
            if base in text: 
                score += 1
            if plural in text: 
                score += 1
        return score

    # Sort documents from highest score to lowest
    unique_docs.sort(key=score_doc, reverse=True)

    # 5. THE BUDGET SAFEGUARD: Only keep the top 3 results
    top_docs = unique_docs[:3]

    # 6. Format with the Strict Rules
    formatted_context = "\n\n---\n\n".join(
        [f"Page {doc.metadata.get('page', 'Unknown')}:\n{doc.page_content}" for doc in top_docs]
    )
    
    return formatted_context

print("🛠️ Tool 'search_internship_history' defined successfully.")
print(f"Tool Description given to LLM: {search_internship_history.description}")

🛠️ Tool 'search_internship_history' defined successfully.
Tool Description given to LLM: Search the Nexteir Internship logs. This is a comprehensive 'Second Brain' 
containing debugging fixes (//problemXX), architectural decisions, 
and useful CLI/Terminal commands (//useful command).


In [5]:
#cell5:
# Initialize the model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Bind the tool to the LLM
tools = [search_internship_history]
llm_with_tools = llm.bind_tools(tools)

# Ask a specific question that requires the tool
test_message = HumanMessage(content="I'm getting an Android layout collapse in my React Native app. How did we fix this during the internship?")
response = llm_with_tools.invoke([test_message])

# Check if the AI decided to use the tool
if response.tool_calls:
    print("🧠 The Agent decided to use a tool!")
    for tool_call in response.tool_calls:
        print(f"-> Tool Selected: {tool_call['name']}")
        print(f"-> Search Query Generated by AI: {tool_call['args']['query']}")
else:
    print("⚠️ The Agent answered directly without using the tool.")
    print(response.content)

🧠 The Agent decided to use a tool!
-> Tool Selected: search_internship_history
-> Search Query Generated by AI: Android layout collapse React Native
